In [1]:
# Create bankruptcy events and eligible nonbankruptcy pseudo-events.
# Place bankruptcy predictions 90 days before filing and require a completed 12-month outcome window.

In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, canonical_security
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/company_availability.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
AVAILABILITY_PATH = PROCESSED / "company_availability.csv"
COMPANY_PATH = PROCESSED / "clean_company_long.csv"
FILING_PATH = ROOT / "data/raw/bankrupt/data.csv"
for path in (AVAILABILITY_PATH, COMPANY_PATH, FILING_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

PREDICTION_LEAD_DAYS = 90
OUTCOME_MONTHS = 12
MIN_FINANCIAL_YEARS = 3
MIN_MARKET_MONTHS = 12
MAX_FINANCIAL_AGE_MONTHS = 18
MAX_MARKET_AGE_DAYS = 45
# Set to a confirmed screening date when known; None uses the conservative
# collection endpoint proxy. The proxy is explicitly reported below.
CONFIRMED_BANKRUPTCY_SCREENING_END = None  # e.g. "2026-05-02"
EVENT_COLUMNS = ["company_id", "prediction_date", "target", "event_type",
                 "prediction_year", "outcome_end_date"]


In [3]:
def canonical_company_id(value):
    text = str(value).strip().upper()
    match = re.fullmatch(r"(SA|NB)_(\d+)(?:\.0)?", text)
    if not match:
        raise ValueError(f"Unrecognized company identifier: {value!r}")
    return f"{match.group(1)}_{int(match.group(2))}"


availability = pd.read_csv(AVAILABILITY_PATH, dtype={"company_id": "string"})
availability["company_id"] = availability["company_id"].map(canonical_company_id)
if availability["company_id"].duplicated().any():
    raise ValueError("company_availability.csv must contain one row per company_id")
availability["company_type"] = availability["company_type"].str.strip().str.lower()
if not availability["company_type"].isin(["bankrupt", "nonbankrupt"]).all():
    raise ValueError("Unexpected company_type")
for column in ["bankruptcy_date", "start_date", "end_date", "first_market_date", "last_market_date"]:
    original = availability[column]
    availability[column] = pd.to_datetime(original, errors="coerce").dt.normalize()
    if (original.notna() & availability[column].isna()).any():
        raise ValueError(f"Invalid dates in {column}")
for column in ["bloomberg_sector", "bloomberg_industry", "bloomberg_identifier"]:
    if availability[column].isna().any() or availability[column].astype(str).str.strip().eq("").any():
        raise ValueError(f"Missing {column}; rerun cleaning before event creation")
availability["security_key"] = canonical_security(availability["bloomberg_identifier"])
availability["ticker_key"] = canonical_security(availability["ticker"])
nonbankrupt_metadata = availability.loc[availability["company_type"].eq("nonbankrupt")].copy()
if nonbankrupt_metadata.empty or nonbankrupt_metadata["end_date"].isna().any():
    raise ValueError("Non-bankrupt collection endpoints are required")
collection_cutoff = nonbankrupt_metadata["end_date"].min()
if CONFIRMED_BANKRUPTCY_SCREENING_END is None:
    outcome_observation_end = collection_cutoff
    print("PROVISIONAL outcome-screening cutoff:", outcome_observation_end.date())
    print("Confirm comprehensive bankruptcy screening through this date before modeling negative targets.")
else:
    outcome_observation_end = pd.Timestamp(CONFIRMED_BANKRUPTCY_SCREENING_END).normalize()
if pd.isna(outcome_observation_end) or outcome_observation_end > pd.Timestamp.today().normalize():
    raise ValueError("Outcome-screening cutoff cannot be missing or in the future")

# Read only the fields that establish usable history; do not load all variables
# or propagate any future values into eligibility or predictors.
history_chunks = []
for chunk in pd.read_csv(COMPANY_PATH, usecols=["company_id", "date", "field", "value"],
                         chunksize=250_000):
    relevant = chunk.loc[chunk["field"].isin(["total_assets", "adjusted_price"])].copy()
    relevant["value"] = pd.to_numeric(relevant["value"], errors="coerce")
    relevant = relevant.loc[relevant["value"].notna() & np.isfinite(relevant["value"])]
    history_chunks.append(relevant)
history = pd.concat(history_chunks, ignore_index=True)
history["company_id"] = history["company_id"].map(canonical_company_id)
history["date"] = pd.to_datetime(history["date"], errors="raise").dt.normalize()
if history.duplicated(["company_id", "date", "field"]).any():
    raise ValueError("Duplicate history records; rerun cleaning")
if not history["company_id"].isin(availability["company_id"]).all():
    raise ValueError("History contains companies absent from company_availability.csv")
assets = history.loc[history["field"].eq("total_assets") & history["value"].ge(0)].copy()
prices = history.loc[history["field"].eq("adjusted_price") & history["value"].gt(0)].copy()
financial_dates = {key: group["date"].sort_values().to_numpy(dtype="datetime64[ns]")
                   for key, group in assets.groupby("company_id", sort=False)}
market_dates = {key: group["date"].sort_values().to_numpy(dtype="datetime64[ns]")
                for key, group in prices.groupby("company_id", sort=False)}
last_market_dates = prices.groupby("company_id")["date"].max()


PROVISIONAL outcome-screening cutoff: 2026-05-02
Confirm comprehensive bankruptcy screening through this date before modeling negative targets.


In [5]:
filings = pd.read_csv(FILING_PATH)
filings["bankruptcy_date"] = pd.to_datetime(filings["bankruptcy_date"], format="mixed", errors="raise").dt.normalize()
if filings["bankruptcy_date"].isna().any():
    raise ValueError("Missing filing dates in bankruptcy registry")
filings["security_key"] = canonical_security(filings["bloomberg_identifier"])
filings["ticker_key"] = canonical_security(filings["ticker"])
# Include retained positive filings even if the separate raw registry differs.
retained_filings = availability.loc[availability["company_type"].eq("bankrupt"),
                                    ["bankruptcy_date", "security_key", "ticker_key"]]
if retained_filings["bankruptcy_date"].isna().any():
    raise ValueError("Every bankrupt company must have a filing date")
filing_lookup = pd.concat([filings[["bankruptcy_date", "security_key", "ticker_key"]],
                           retained_filings], ignore_index=True).drop_duplicates()
security_first_filing = filing_lookup.dropna(subset=["security_key"]).groupby("security_key")["bankruptcy_date"].min()
ticker_first_filing = filing_lookup.dropna(subset=["ticker_key"]).groupby("ticker_key")["bankruptcy_date"].min()
nonbankrupt_metadata["known_first_filing"] = pd.concat([
    nonbankrupt_metadata["security_key"].map(security_first_filing),
    nonbankrupt_metadata["ticker_key"].map(ticker_first_filing)
], axis=1).min(axis=1)

EMPTY_DATES = np.array([], dtype="datetime64[ns]")
def history_at_predictions(company_id, prediction_dates):
    predictions = pd.DatetimeIndex(prediction_dates)
    if predictions.hasnans:
        raise ValueError("Missing prediction dates")
    query = predictions.to_numpy(dtype="datetime64[ns]")
    f_dates = financial_dates.get(company_id, EMPTY_DATES)
    m_dates = market_dates.get(company_id, EMPTY_DATES)
    # side='left' enforces strict observation_date < prediction_date.
    f_count = np.searchsorted(f_dates, query, side="left")
    m_count = np.searchsorted(m_dates, query, side="left")
    last_financial = np.full(len(query), np.datetime64("NaT", "ns"), dtype="datetime64[ns]")
    last_market = last_financial.copy()
    if len(f_dates):
        has_f = f_count > 0
        last_financial[has_f] = f_dates[f_count[has_f] - 1]
        # The first observed date in each calendar year is known at that date.
        f_years = f_dates.astype("datetime64[Y]")
        first_in_year = np.r_[True, f_years[1:] != f_years[:-1]]
        years_before = np.searchsorted(f_dates[first_in_year], query, side="left")
    else:
        years_before = np.zeros(len(query), dtype=int)
    if len(m_dates):
        has_m = m_count > 0
        last_market[has_m] = m_dates[m_count[has_m] - 1]
    # Previous calendar months are entirely before the prediction date.
    required_months = (query.astype("datetime64[M]").astype(np.int64)[:, None]
                       - np.arange(1, MIN_MARKET_MONTHS + 1)[None, :])
    observed_months = np.unique(m_dates.astype("datetime64[M]").astype(np.int64))
    recent_market_months = np.isin(required_months, observed_months).sum(axis=1)
    oldest_acceptable_financial = (predictions - pd.DateOffset(months=MAX_FINANCIAL_AGE_MONTHS)).to_numpy()
    oldest_acceptable_market = (predictions - pd.Timedelta(days=MAX_MARKET_AGE_DAYS)).to_numpy()
    result = pd.DataFrame({
        "company_id": company_id, "prediction_date": predictions,
        "financial_years_before": years_before,
        "recent_market_months_before": recent_market_months,
        "last_financial_before": last_financial,
        "last_market_before": last_market,
        "enough_financial_history": years_before >= MIN_FINANCIAL_YEARS,
        "fresh_financial_history": last_financial >= oldest_acceptable_financial,
        "enough_market_history": recent_market_months == MIN_MARKET_MONTHS,
        "recent_market_observation": last_market >= oldest_acceptable_market,
    })
    result["history_eligible"] = result[["enough_financial_history", "fresh_financial_history",
                                        "enough_market_history", "recent_market_observation"]].all(axis=1)
    return result


In [6]:
bankrupt_audit = availability.loc[availability["company_type"].eq("bankrupt"),
                                  ["company_id", "bankruptcy_date", "end_date"]].copy()
bankrupt_audit["prediction_date"] = bankrupt_audit["bankruptcy_date"] - pd.Timedelta(days=PREDICTION_LEAD_DAYS)
bankrupt_audit["outcome_end_date"] = bankrupt_audit["prediction_date"] + pd.DateOffset(months=OUTCOME_MONTHS)
bankrupt_history = pd.concat([
    history_at_predictions(row.company_id, [row.prediction_date])
    for row in bankrupt_audit.itertuples(index=False)
], ignore_index=True)
bankrupt_audit = bankrupt_audit.merge(bankrupt_history, on=["company_id", "prediction_date"],
                                      how="left", validate="one_to_one")
bankrupt_audit["outcome_complete"] = bankrupt_audit["outcome_end_date"].le(outcome_observation_end)
bankrupt_audit["filing_in_outcome"] = (
    bankrupt_audit["bankruptcy_date"].gt(bankrupt_audit["prediction_date"])
    & bankrupt_audit["bankruptcy_date"].le(bankrupt_audit["outcome_end_date"]))
bankrupt_audit["eligible"] = bankrupt_audit[["history_eligible", "outcome_complete", "filing_in_outcome"]].all(axis=1)
bankruptcy_events = bankrupt_audit.loc[bankrupt_audit["eligible"]].copy()
bankruptcy_events["target"] = 1
bankruptcy_events["event_type"] = "bankruptcy"
bankruptcy_events["prediction_year"] = bankruptcy_events["prediction_date"].dt.year
bankruptcy_events = bankruptcy_events[EVENT_COLUMNS].sort_values(["prediction_date", "company_id"]).reset_index(drop=True)
candidate_prediction_dates = pd.DatetimeIndex(bankruptcy_events["prediction_date"].drop_duplicates().sort_values())
if candidate_prediction_dates.empty:
    raise ValueError("No eligible bankrupt event dates: inspect history thresholds and outcome cutoff")


In [7]:
candidate_parts = []
for row in nonbankrupt_metadata.itertuples(index=False):
    part = history_at_predictions(row.company_id, candidate_prediction_dates)
    part["outcome_end_date"] = part["prediction_date"] + pd.DateOffset(months=OUTCOME_MONTHS)
    part["outcome_complete"] = part["outcome_end_date"].le(outcome_observation_end)
    # Future price coverage is used only to establish outcome follow-up.
    # Also respect the declared company-specific extraction endpoint.
    followup_end = last_market_dates.get(row.company_id, pd.NaT)
    part["company_followup_complete"] = (part["outcome_end_date"].le(followup_end)
                                        & part["outcome_end_date"].le(row.end_date))
    part["no_known_filing_through_outcome"] = (
        pd.isna(row.known_first_filing) | part["outcome_end_date"].lt(row.known_first_filing))
    part["eligible"] = part[["history_eligible", "outcome_complete", "company_followup_complete",
                             "no_known_filing_through_outcome"]].all(axis=1)
    candidate_parts.append(part)
candidate_audit = pd.concat(candidate_parts, ignore_index=True)
nonbankruptcy_candidate_events = candidate_audit.loc[candidate_audit["eligible"]].copy()
nonbankruptcy_candidate_events["target"] = 0
nonbankruptcy_candidate_events["event_type"] = "nonbankruptcy_candidate"
nonbankruptcy_candidate_events["prediction_year"] = nonbankruptcy_candidate_events["prediction_date"].dt.year
nonbankruptcy_candidate_events = nonbankruptcy_candidate_events[EVENT_COLUMNS].sort_values(
    ["prediction_date", "company_id"]).reset_index(drop=True)
checks = ["enough_financial_history", "fresh_financial_history", "enough_market_history",
          "recent_market_observation", "outcome_complete", "company_followup_complete",
          "no_known_filing_through_outcome"]


In [8]:
# Write both event tables.

PROCESSED.mkdir(parents=True, exist_ok=True)
for filename, frame in [("bankruptcy_events.csv", bankruptcy_events),
                        ("nonbankruptcy_candidate_events.csv", nonbankruptcy_candidate_events)]:
    output_path = PROCESSED / filename
    frame.to_csv(output_path, index=False, date_format="%Y-%m-%d")

coverage = pd.concat([bankruptcy_events, nonbankruptcy_candidate_events]).groupby(
    ["prediction_year", "event_type"]).size().unstack(fill_value=0)
